# 02 - ViT-GPT-2 Captioning → Emotion/NER/POS → Paired Train/Test Split

End-to-end pipeline following the architecture:

```
anime_256 images
      ↓
 ViT-GPT-2 captioner  →  caption_raw.csv
      ↓
 NER / POS / Emotion  →  caption.csv  (file, caption, emotion, ner_cues, pos_cues)
      ↓
 Pair by filename + Split (seed 42, 80/20)
      ↓                    ↓
 data/train/ + train.csv   data/test/ + test.csv
```

**Chain in :** `data/processed/anime_256/`
**Chain out:** `artifacts/caption.csv`, `artifacts/train.csv`, `artifacts/test.csv`, `data/train/`, `data/test/`

In [3]:

# Mount Google Drive
!fusermount -u /content/drive 2>/dev/null
!rm -rf /content/drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


---
## SECTION 1 — Environment Setup

In [4]:
import os, glob, json, math, textwrap, random, re, shutil
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE  = torch.float16 if DEVICE == 'cuda' else torch.float32

# ROOT: first Drive folder that exists
ROOT = next((r for r in [
    '/content/drive/MyDrive/Text2ImageNarration',
    '/content/drive/MyDrive/AnimeStoryGen',
    '.'
] if os.path.isdir(r)), '.')

DATA      = f'{ROOT}/data'
PROC      = f'{ROOT}/data/processed/anime_256'
ART       = f'{ROOT}/artifacts'
CKPT      = f'{ROOT}/checkpoints'
GEN       = f'{ROOT}/outputs/generated'
EVALD     = f'{ROOT}/outputs/eval'
TRAIN_DIR = f'{ROOT}/data/train'
TEST_DIR  = f'{ROOT}/data/test'

for d in (DATA, ART, CKPT, GEN, EVALD, TRAIN_DIR, TEST_DIR):
    os.makedirs(d, exist_ok=True)

SEED       = 42
N_TEST     = 200
TEST_RATIO = 0.20
IMAGE_EXTS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')

def img_dir():
    if os.path.isdir(PROC):
        imgs = [f for f in glob.glob(f'{PROC}/**/*', recursive=True)
                if f.lower().endswith(IMAGE_EXTS)]
        if imgs:
            return PROC
    fallback = f'{DATA}/anime_images'
    return fallback if os.path.isdir(fallback) else PROC

from PIL import Image, ImageDraw, ImageFont

src  = img_dir()
imgs = [f for f in glob.glob(f'{src}/**/*', recursive=True)
        if f.lower().endswith(IMAGE_EXTS)]
print(f'Device       : {DEVICE}')
print(f'ROOT         : {ROOT}')
print(f'Image source : {src}')
print(f'Images found : {len(imgs)}')
if imgs:
    print(f'Sample files : {[os.path.basename(f) for f in imgs[:5]]}')
else:
    print(f'WARNING: No images found — check your Drive path: {PROC}')

Device       : cuda
ROOT         : /content/drive/MyDrive/Text2ImageNarration
Image source : /content/drive/MyDrive/Text2ImageNarration/data/processed/anime_256
Images found : 55
Sample files : ['1762.jpg', '1763.jpg', '1764.jpg', '1765.jpg', '1766.jpg']


In [5]:
import sys, subprocess, importlib
for p in ['transformers']:
    if importlib.util.find_spec(p.split('==')[0]) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p])
print('deps ready')

deps ready


---
## SECTION 2 — ViT-GPT-2 Captioning → `caption_raw.csv`

In [6]:
from transformers import VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer
import csv

VID   = 'nlpconnect/vit-gpt2-image-captioning'
vproc = ViTImageProcessor.from_pretrained(VID)
vtok  = AutoTokenizer.from_pretrained(VID)
vmdl  = VisionEncoderDecoderModel.from_pretrained(VID).to(DEVICE).eval()
RAW   = f'{ART}/caption_raw.csv'

@torch.no_grad()
def caption(img):
    if isinstance(img, str): img = Image.open(img).convert('RGB')
    pix = vproc(img, return_tensors='pt').pixel_values.to(DEVICE)
    return vtok.decode(
        vmdl.generate(pix, max_length=32, num_beams=4)[0],
        skip_special_tokens=True
    )

def done_set(p):
    s = {}
    if os.path.isfile(p):
        for r in csv.DictReader(open(p, newline='')):
            if r.get('file'): s[r['file']] = r.get('caption', '')
    return s

def caption_all(limit=None):
    src = img_dir()
    files = sorted(
        f for f in glob.glob(f'{src}/**/*', recursive=True)
        if f.lower().endswith(IMAGE_EXTS)
    )
    if not files:
        print(f'No images found in {src}. Check your Drive path.')
        return
    if limit: files = files[:limit]
    have = done_set(RAW)
    todo = [f for f in files if os.path.basename(f) not in have]
    print(f'{len(have)} done | {len(todo)} remaining | {len(files)} total')
    new = not os.path.isfile(RAW)
    fh  = open(RAW, 'a', newline='')
    w   = csv.DictWriter(fh, fieldnames=['file', 'caption'])
    if new: w.writeheader(); fh.flush()
    n = 0
    try:
        for f in todo:
            try: cap = caption(f)
            except Exception as e: print('skip', f, e); continue
            w.writerow({'file': os.path.basename(f), 'caption': cap})
            n += 1
            fh.flush(); os.fsync(fh.fileno())
    finally:
        fh.flush(); fh.close()
    print(f'captioned {n} new -> {RAW}')

# Set limit=None for the full dataset
caption_all(limit=50)

preprocessor_config.json:   0%|          | 0.00/228 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/120 [00:00<?, ?B/s]

...

The tied weights mapping and config for this model specifies to tie decoder.transformer.wte.weight to decoder.lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
VisionEncoderDecoderModel LOAD REPORT from: nlpconnect/vit-gpt2-image-captioning
Key                                                       | Status     |  | 
----------------------------------------------------------+------------+--+-
decoder.transformer.h.{0...11}.attn.bias                  | UNEXPECTED |  | 
decoder.transformer.h.{0...11}.crossattention.bias        | UNEXPECTED |  | 
decoder.transformer.h.{0...11}.crossattention.masked_bias | UNEXPECTED |  | 
decoder.transformer.h.{0...11}.attn.masked_bias           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/982M [00:00<?, ?B/s]

0 done | 50 remaining | 50 total


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
You may ignore this warning if your `pad_token_id` (50256) is identical to the `bos_token_id` (50256), `eos_token_id` (50256), or the `sep_token_id` (None), and your input is not padded.


captioned 50 new -> /content/drive/MyDrive/Text2ImageNarration/artifacts/caption_raw.csv


---
## SECTION 3 — Emotion Tagging → `caption.csv`

In [7]:
import pandas as pd
from transformers import pipeline as hf_pipeline

try:
    clf = hf_pipeline(
        'text-classification',
        model='j-hartmann/emotion-english-distilroberta-base',
        top_k=1, device=0 if DEVICE == 'cuda' else -1
    )
    def tag(t):
        try: return clf(str(t)[:256])[0][0]['label'].lower()
        except Exception: return 'neutral'
except Exception as e:
    print('emotion classifier unavailable, defaulting neutral:', e)
    tag = lambda t: 'neutral'

raw = pd.read_csv(RAW)
if len(raw) == 0:
    raise ValueError(f'caption_raw.csv is empty at {RAW}. Re-run the captioning cell.')

raw['emotion'] = [tag(c) for c in raw['caption']]
raw.to_csv(f'{ART}/caption.csv', index=False)
print('wrote', f'{ART}/caption.csv', '| emotions:', raw['emotion'].value_counts().to_dict())

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/294 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


wrote /content/drive/MyDrive/Text2ImageNarration/artifacts/caption.csv | emotions: {'disgust': 36, 'fear': 14}


---
## SECTION 4 — NER / POS Cue Extraction

In [8]:
import sys, subprocess
try:
    import spacy
except Exception:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'spacy'])
    import spacy
try:
    nlp = spacy.load('en_core_web_sm')
except Exception:
    subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'])
    nlp = spacy.load('en_core_web_sm')

import pandas as pd
df = pd.read_csv(f'{ART}/caption.csv')
if len(df) == 0:
    raise ValueError('caption.csv is empty — run emotion tagging cell first.')

def cues(text):
    doc  = nlp(str(text))
    ents = '|'.join(f'{e.text}:{e.label_}' for e in doc.ents)
    pos  = '|'.join(t.text for t in doc if t.pos_ in ('ADJ', 'VERB', 'NOUN'))[:200]
    return ents, pos

pairs = [cues(c) for c in df['caption']]
df['ner_cues'] = [e for e, _ in pairs]
df['pos_cues'] = [p for _, p in pairs]
df.to_csv(f'{ART}/caption.csv', index=False)

print('added NER/POS columns -> caption.csv')
print(f'  Total rows : {len(df)}')
print(f'  Columns    : {df.columns.tolist()}')
print(f'  Sample NER : {df["ner_cues"].iloc[0]}')
print(f'  Sample POS : {df["pos_cues"].iloc[0]}')

added NER/POS columns -> caption.csv
  Total rows : 50
  Columns    : ['file', 'caption', 'emotion', 'ner_cues', 'pos_cues']
  Sample NER : 
  Sample POS : painting|woman|black|white|outfit


---
## SECTION 5 — Paired Train / Test Split by Filename (seed 42)

In [9]:
import pandas as pd

# Load caption.csv
df = pd.read_csv(f'{ART}/caption.csv')
if len(df) == 0:
    raise ValueError('caption.csv is empty — re-run NER/POS cell.')

# Build filename -> full path map from anime_256
src = img_dir()
all_image_files = [f for f in glob.glob(f'{src}/**/*', recursive=True)
                   if f.lower().endswith(IMAGE_EXTS)]
image_map = {os.path.basename(f): f for f in all_image_files}

print(f'caption.csv rows    : {len(df)}')
print(f'Images in anime_256 : {len(image_map)}')

# Inner join — keep only rows that have BOTH a caption AND an image
df['_has_image'] = df['file'].map(lambda f: f in image_map)
missing_imgs = df[~df['_has_image']]['file'].tolist()
missing_caps = [f for f in image_map if f not in set(df['file'])]
df = df[df['_has_image']].drop(columns=['_has_image']).reset_index(drop=True)

print(f'Paired rows         : {len(df)}')
if missing_imgs: print(f'  Captions missing image ({len(missing_imgs)}): {missing_imgs[:3]}')
if missing_caps: print(f'  Images missing caption ({len(missing_caps)}): {missing_caps[:3]}')

# Sort by filename -> shuffle with seed 42 -> split
df = df.sort_values('file').reset_index(drop=True)
df_shuffled = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

n_test  = min(N_TEST, max(1, int(len(df_shuffled) * TEST_RATIO)))
n_train = len(df_shuffled) - n_test

test_df  = df_shuffled.iloc[:n_test].reset_index(drop=True)
train_df = df_shuffled.iloc[n_test:].reset_index(drop=True)

print(f'\nSplit (seed {SEED}, ratio {1-TEST_RATIO:.0%}/{TEST_RATIO:.0%})')
print(f'  Train : {len(train_df)} pairs')
print(f'  Test  : {len(test_df)}  pairs')

caption.csv rows    : 50
Images in anime_256 : 55
Paired rows         : 50
  Images missing caption (5): ['1995.jpg', '1996.jpg', '1997.jpg']

Split (seed 42, ratio 80%/20%)
  Train : 40 pairs
  Test  : 10  pairs


In [10]:
# Save train.csv & test.csv
train_df.to_csv(f'{ART}/train.csv', index=False)
test_df.to_csv(f'{ART}/test.csv',   index=False)

print(f'Saved: {ART}/train.csv  ({len(train_df)} rows)')
print(f'Saved: {ART}/test.csv   ({len(test_df)} rows)')
print('\nColumns:', train_df.columns.tolist())
print('\nSample train rows:')
print(train_df[['file', 'caption', 'emotion']].head(3).to_string(index=False))
print('\nSample test rows:')
print(test_df[['file', 'caption', 'emotion']].head(3).to_string(index=False))

Saved: /content/drive/MyDrive/Text2ImageNarration/artifacts/train.csv  (40 rows)
Saved: /content/drive/MyDrive/Text2ImageNarration/artifacts/test.csv   (10 rows)

Columns: ['file', 'caption', 'emotion', 'ner_cues', 'pos_cues']

Sample train rows:
    file                                          caption emotion
1790.jpg         a doll with a face painted on it's face  disgust
1766.jpg           a painting of a woman in a red outfit  disgust
1920.jpg a painting of a person with a cat on their head  disgust

Sample test rows:
    file                                                caption emotion
1791.jpg        a painting of a woman holding a stuffed animal  disgust
1922.jpg a close up of a picture of a girl with a cartoon face  disgust
1879.jpg           a close up picture of a girl with pink hair     fear


In [11]:
# Copy paired images to data/train/ and data/test/
def copy_images(caption_df, dest_dir, label):
    copied, skipped, missing = 0, 0, []
    for fname in caption_df['file']:
        src_path = image_map.get(fname)
        if src_path is None:
            missing.append(fname); continue
        dst_path = os.path.join(dest_dir, fname)
        if os.path.exists(dst_path):
            skipped += 1; continue
        shutil.copy2(src_path, dst_path)
        copied += 1
        if copied % 200 == 0:
            print(f'  {label}: {copied} copied...', end='\r')
    print(f'  {label}: {copied} copied | {skipped} already existed | {len(missing)} missing source')

print('Copying train images...')
copy_images(train_df, TRAIN_DIR, 'TRAIN')
print('Copying test images...')
copy_images(test_df,  TEST_DIR,  'TEST ')

print(f'\ndata/train/ : {len(os.listdir(TRAIN_DIR))} images')
print(f'data/test/  : {len(os.listdir(TEST_DIR))} images')

Copying train images...
  TRAIN: 40 copied | 0 already existed | 0 missing source
Copying test images...
  TEST : 10 copied | 0 already existed | 0 missing source

data/train/ : 40 images
data/test/  : 10 images


In [12]:
# Verification
train_names       = set(train_df['file'])
test_names        = set(test_df['file'])
overlap           = train_names & test_names
train_on_disk     = set(os.listdir(TRAIN_DIR))
test_on_disk      = set(os.listdir(TEST_DIR))
train_miss_disk   = train_names - train_on_disk
test_miss_disk    = test_names  - test_on_disk

print('══════════════ Verification ══════════════')
print(f'Train — CSV rows : {len(train_df)}  | Images on disk : {len(train_on_disk)}')
print(f'Test  — CSV rows : {len(test_df)}   | Images on disk : {len(test_on_disk)}')
print(f'Overlap (must=0) : {len(overlap)}')
print(f'Train missing on disk : {len(train_miss_disk)}')
print(f'Test  missing on disk : {len(test_miss_disk)}')

if not overlap and not train_miss_disk and not test_miss_disk:
    print('\n✅ Perfect — all images paired with captions, zero overlap.')
else:
    if overlap:         print(f'\nOverlap: {list(overlap)[:5]}')
    if train_miss_disk: print(f'Train images missing on disk: {list(train_miss_disk)[:5]}')
    if test_miss_disk:  print(f'Test  images missing on disk: {list(test_miss_disk)[:5]}')

print('\n══════════════ Final Outputs ══════════════')
print(f'  artifacts/caption.csv  : {len(df)} rows | {df.columns.tolist()}')
print(f'  artifacts/train.csv    : {len(train_df)} rows')
print(f'  artifacts/test.csv     : {len(test_df)} rows')
print(f'  data/train/            : {len(train_on_disk)} images')
print(f'  data/test/             : {len(test_on_disk)} images')
print('\nNext -> 03_Vaswani_BaseLM (trains on train.csv)')

══════════════ Verification ══════════════
Train — CSV rows : 40  | Images on disk : 40
Test  — CSV rows : 10   | Images on disk : 10
Overlap (must=0) : 0
Train missing on disk : 0
Test  missing on disk : 0

✅ Perfect — all images paired with captions, zero overlap.

══════════════ Final Outputs ══════════════
  artifacts/caption.csv  : 50 rows | ['file', 'caption', 'emotion', 'ner_cues', 'pos_cues']
  artifacts/train.csv    : 40 rows
  artifacts/test.csv     : 10 rows
  data/train/            : 40 images
  data/test/             : 10 images

Next -> 03_Vaswani_BaseLM (trains on train.csv)
